# 1. Data Ingestion

What the brief asks for, and where each part is answered in this notebook:

| Brief | Section |
|---|---|
| Load and join all 5 tables (document your join logic / keys used) | 1.1 |
| Handle missing values, inconsistent labels, and other data quality issues: state how, not just "dropped nulls" | 1.2 |
| Output: one clean, analysis-ready dataset (+ a short data quality notes section) | 1.3 |

The code lives in `src/ingest.py`; this notebook runs it and shows the results. Full notes:
`docs/data_quality_notes.md`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import nbstyle  # puts src/ on the path
from config import RAW, PROCESSED
import ingest
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 130)

raw = dict(zip(["customers", "loan_accounts", "bureau_credit", "repayment_history", "collections_events"],
               ingest.load_raw()))
pd.DataFrame([{"table": name, "rows": len(df), "columns": df.shape[1]} for name, df in raw.items()])

,table,rows,columns
0,customers,10000,15
1,loan_accounts,14685,11
2,bureau_credit,22979,10
3,repayment_history,176220,7
4,collections_events,8635,6


## 1.1 Join logic and keys

```
customers ──1:N── loan_accounts ──1:12── repayment_history
    │                   └──────1:N────── collections_events
    └──1:N── bureau_credit
```

| Table | One row per | Key | Joined to the output on |
|---|---|---|---|
| loan_accounts | loan account | `account_id` | the spine: one output row per loan, holds `target_default` |
| customers | customer | `customer_id` | `customer_id`, many-to-one |
| bureau_credit | external credit line | `bureau_id` | aggregated to one row per customer first, then `customer_id` |
| repayment_history | account x month (1–12) | `account_id` + `payment_month` | aggregated to one row per account first, then `account_id` |
| collections_events | collections action | `collection_event_id` | aggregated to one row per account first, then `account_id` (dashboard only) |

Raw tables are never joined directly: a customer with 3 loans and 5 bureau lines would become 3 x 5 x 12 = 180 rows and
count the same exposure many times. Each join is validated (`validate="m:1"` / `"1:1"`) and the row count is asserted
to stay at 14,685. The key checks:

In [2]:
c, l, b, r, k = raw.values()
pd.DataFrame([
    ("customer_id unique in customers", c.customer_id.is_unique),
    ("account_id unique in loan_accounts", l.account_id.is_unique),
    ("every loan has a customer", l.customer_id.isin(c.customer_id).all()),
    ("every bureau line has a customer", b.customer_id.isin(c.customer_id).all()),
    ("every repayment row has a loan", r.account_id.isin(l.account_id).all()),
    ("every collections event has a loan", k.account_id.isin(l.account_id).all()),
    ("one row per account and month in repayment", not r.duplicated(["account_id", "payment_month"]).any()),
    ("customers with no bureau record (thin file)", int((~c.customer_id.isin(b.customer_id)).sum())),
], columns=["check", "result"])

,check,result
0,customer_id unique in customers,True
1,account_id unique in loan_accounts,True
2,every loan has a customer,True
3,every bureau line has a customer,True
4,every repayment row has a loan,True
5,every collections event has a loan,True
6,one row per account and month in repayment,True
7,customers with no bureau record (thin file),1008


## 1.2 Data quality issues, and how each was handled

Running the pipeline. Every check is counted and logged, including the ones that pass:

In [3]:
ingest.main()

accounts_clean.csv: 14,685 rows x 85 columns, default rate 16.1%
dq_log.csv: 45 checks, 30 with issues

             table                     column  rows_affected  pct_of_rows                                                                                                                                                                          issue
         customers                customer_id              0         0.00                                                                                                                                                          duplicate primary key
     loan_accounts                 account_id              0         0.00                                                                                                                                                          duplicate primary key
     bureau_credit                  bureau_id              0         0.00                                                                            

In [4]:
dq = pd.read_csv(PROCESSED / "dq_log.csv")
issues = dq[dq.action != "check passed"]
print(f"{len(dq)} checks run: {len(dq) - len(issues)} passed, {len(issues)} found an issue that was fixed or flagged")
issues[["table", "column", "issue", "rows_affected", "pct_of_rows", "action"]]

45 checks run: 15 passed, 30 found an issue that was fixed or flagged


,table,column,issue,rows_affected,pct_of_rows,action
9,customers,gender,inconsistent labels 'M' / 'F',600,6.00,mapped to 'Male' / 'Female'
10,customers,years_employed,missing for Unemployed customers,23,0.23,set to 0 (every other Unemployed customer has 0)
11,customers,years_employed,longer than age - 16 (impossible work history),726,7.26,"capped at age - 16, flagged"
12,customers,years_employed,missing,477,4.77,"median of the same employment type, flagged"
13,customers,num_dependents,missing,500,5.00,"median (1), flagged"
14,customers,years_at_residence,missing,500,5.00,"median (2.8), flagged"
15,customers,years_at_residence,longer than the customer's age,5,0.05,capped at age
16,customers,annual_income,"sits exactly at the 12,000 minimum (lower values look clipped)",76,0.76,"kept, flagged"
17,customers,annual_income,"Unemployed but earning above the employed median (80,700)",22,0.22,kept (could be pension or rental income); needs income verification
18,customers,customer_since,later than the customer's own first loan,5673,56.73,unreliable: relationship_start = earlier of the two dates; not used in the model


A few of the fixes, before and after:

In [5]:
clean = pd.read_csv(PROCESSED / "accounts_clean.csv")
customers_clean = pd.read_csv(PROCESSED / "customers_clean.csv")
print("Gender labels before:", raw["customers"].gender.value_counts().to_dict())
print("Gender labels after: ", customers_clean.gender.value_counts().to_dict())
print()
print("Missing interest rates before:", int(raw["loan_accounts"].interest_rate_pct.isna().sum()),
      "| after:", int(clean.interest_rate_pct.isna().sum()),
      "| recovered from the instalment formula and flagged:", int(clean.interest_rate_derived.sum()))
print("Missing years employed before:", int(raw["customers"].years_employed.isna().sum()),
      "| after:", int(customers_clean.years_employed.isna().sum()),
      "| capped impossible values:", int(customers_clean.years_employed_capped.sum()))
print("Rows dropped: none (loan accounts in = out:", len(raw["loan_accounts"]), "=", len(clean), ")")

Gender labels before: {'Male': 5341, 'Female': 4059, 'M': 358, 'F': 242}
Gender labels after:  {'Male': 5699, 'Female': 4301}

Missing interest rates before: 294 | after: 0 | recovered from the instalment formula and flagged: 294
Missing years employed before: 500 | after: 0 | capped impossible values: 726
Rows dropped: none (loan accounts in = out: 14685 = 14685 )


## 1.3 Output: one clean, analysis-ready dataset

`data/processed/accounts_clean.csv`: one row per loan account, every table's information joined in, with flags that
record each fix.

In [6]:
print(clean.shape)
groups = {
    "ids and target": ["account_id", "customer_id", "target_default"],
    "loan": [c for c in clean.columns if c in ("loan_type", "loan_amount", "exposure", "interest_rate_pct", "tenure_months",
                                                "emi", "origination_date", "disbursal_channel")],
    "bureau summary": [c for c in clean.columns if c.startswith("bureau_")],
    "repayment summary": [c for c in clean.columns if c.startswith("rep_")],
    "collections summary (dashboard only)": [c for c in clean.columns if c.startswith("coll_")],
    "data quality flags": [c for c in clean.columns if c.endswith(("_imputed", "_capped", "_at_floor", "_derived",
                                                                 "_mismatch", "_after_first_loan", "_exceeds_income"))],
}
display(pd.DataFrame([{"group": g, "columns": len(cols), "examples": ", ".join(cols[:6])} for g, cols in groups.items()]))
clean.head(3)

(14685, 85)


,group,columns,examples
0,ids and target,3,"account_id, customer_id, target_default"
1,loan,8,"loan_type, loan_amount, exposure, interest_rate_pct, tenure_months, emi"
2,bureau summary,15,"bureau_n_lines, bureau_n_active, bureau_n_closed, bureau_n_written_off, bureau_n_dpd_30_plus, bureau_total_sanctioned"
3,repayment summary,15,"rep_n_on_time, rep_n_late, rep_n_missed, rep_max_dpd, rep_mean_dpd, rep_mean_dpd_first3"
4,collections summary (dashboard only),6,"coll_n_events_raw, coll_n_events, coll_total_recovered, coll_last_action, coll_last_outcome, coll_last_event_date"
5,data quality flags,10,"interest_rate_derived, loan_amount_at_floor, purpose_product_mismatch, years_employed_capped, years_employed_imputed, num_depe..."


,account_id,customer_id,target_default,loan_type,loan_purpose,loan_amount,exposure,interest_rate_pct,tenure_months,emi,...,coll_last_event_date,bureau_no_file,emi_to_income,emi_exceeds_income,emi_to_income_band,loan_to_income,lti_band,bureau_debt_to_income,n_loans_customer,customer_total_exposure
0,ACC200001,CUST100000,1,Mortgage,Wedding,306310.51,306310.51,6.06,360,1848.32,...,NaN,1,0.252906,0,20-30%,3.492708,2x+,0.000000,1,306310.51
1,ACC200002,CUST100001,0,Auto Loan,Home Improvement,33921.61,33921.61,5.52,60,648.26,...,2023-06-20,0,0.648260,0,50%+,2.826801,2x+,4.992360,1,33921.61
2,ACC200003,CUST100002,0,Personal Loan,Business Expansion,4469.93,4469.93,10.98,48,115.48,...,NaN,0,0.023058,0,<10%,0.074375,<0.25x,0.582345,1,4469.93


### Data quality notes (short)

- **Join logic and keys:** `loan_accounts` is the spine (one row per loan, 14,685 rows). `customers` joins on
  `customer_id`; `bureau_credit` and `repayment_history` are aggregated to one row per customer and per account, then
  joined; `collections_events` is aggregated per account for the dashboard only. Row count is asserted unchanged.
- **Missing values, and how:** years employed (500): 0 for the 23 unemployed, median of the same employment type for
  the rest, flagged. Dependents and years at address (500 each): median, flagged. Interest rate (294): recovered
  exactly from the instalment, amount and tenure. Bureau days past due (689): 0 on closed lines, median of the same
  status otherwise. 1,008 customers with no bureau record: flagged as thin file, not filled with zeros.
- **Inconsistent labels:** gender "M" / "F" for 600 customers, mapped to "Male" / "Female".
- **Other issues:** 726 impossible job tenures (capped at age − 16); `customer_since` later than the first loan for 57%
  of customers (not used); bureau dates after the as-of date (not used); 45% of collections events dated before their
  loan existed (collections kept out of the model); balances that don't follow the payment schedule (exposure =
  original loan amount); 79 loans whose instalment exceeds income (flagged).
- **Nothing was silently dropped:** every row is kept, and all 45 checks are logged with the rows they affected.